# Qwen3 1.7B Local Test Baseline

Closed-book baseline evaluation for `Qwen/Qwen3-1.7B` on all 60 test problems.

## Setup

Run this locally from VS Code. The model downloads into the Hugging Face cache, not this repository.

In [1]:
!pip install -q -U transformers accelerate pandas tqdm python-dotenv

In [2]:
from pathlib import Path
import os
import sys

import torch
from dotenv import load_dotenv
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
PROJECT_ROOT = Path("/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune")
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")
PROJECT_ROOT

PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune')

In [4]:
from training_eval.eval_utils import (
    GRADING_POLICY,
    default_test_dir,
    extract_answer,
    extract_json_object,
    is_correct,
    load_jsonl_records,
    make_closed_book_prompt,
    rows_to_frame,
    save_results,
    summarize_accuracy,
)

## Load Dev Records

In [5]:
DATA_DIR = default_test_dir(PROJECT_ROOT)
records = load_jsonl_records(DATA_DIR, pattern="*_preview.jsonl")
len(records), DATA_DIR

(60,
 PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/benchmark/data/test'))

## Load Model

Qwen3 defaults to a thinking-style chat mode. The generation helper below sets `enable_thinking=False` so the model has a much better chance of returning the requested final JSON inside the token budget.


In [6]:
MODEL_NAME = "Qwen/Qwen3-1.7B"
HF_TOKEN = os.environ.get("HF_TOKEN")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
dtype = torch.float16 if device.type == "mps" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=HF_TOKEN, torch_dtype=dtype)
model.to(device)
model.eval()

device

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 311/311 [00:03<00:00, 84.09it/s] 


device(type='mps')

In [7]:
MAX_NEW_TOKENS = 256


def generate_answer(problem):
    messages = [{"role": "user", "content": make_closed_book_prompt(problem)}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


## One-Example Smoke Test

In [8]:
raw_output = generate_answer(records[0]["problem"])
predicted = extract_answer(raw_output, records[0]["canonical_answer"])
raw_output, predicted, records[0]["canonical_answer"], is_correct(predicted, records[0]["canonical_answer"])

('{"expected_time": "2"}',
 {'expected_time': '2'},
 {'expected_time': '21'},
 False)

## Run Full Test Evaluation

In [9]:
rows = []

for record in tqdm(records):
    raw_output = generate_answer(record["problem"])
    predicted = extract_answer(raw_output, record["canonical_answer"])
    metadata = record.get("metadata", {})

    rows.append({
        "id": record["id"],
        "family": record["family"],
        "problem_type": record["problem_type"],
        "difficulty": record["difficulty"],
        "manual_variation": metadata.get("manual_variation", False),
        "manual_problem_variation": metadata.get("manual_problem_variation", False),
        "manual_reasoning_variation": metadata.get("manual_reasoning_variation", False),
        "problem": record["problem"],
        "canonical_answer": record["canonical_answer"],
        "raw_output": raw_output,
        "predicted_answer": predicted,
        "correct": is_correct(predicted, record["canonical_answer"]),
    })

df = rows_to_frame(rows)
df.head()

100%|██████████| 60/60 [00:38<00:00,  1.57it/s]


,id,family,problem_type,difficulty,manual_variation,manual_problem_variation,manual_reasoning_variation,problem,canonical_answer,raw_output,predicted_answer,correct,answer_type
0,hitting_time_expectation_dev_004100,hitting_time_expectation,symmetric_boundaries_zero_a,1,True,True,False,Start a fair random walk at 3. Stop the first ...,{'expected_time': '21'},"{""expected_time"": ""2""}",{'expected_time': '2'},False,non_binary
1,hitting_time_expectation_dev_004101,hitting_time_expectation,symmetric_boundaries_zero_a,1,True,True,True,A fair nearest-neighbor walk starts at 3 and i...,{'expected_time': '6'},"{""expected_time"": ""2.5""}",{'expected_time': '2.5'},False,non_binary
2,hitting_time_expectation_dev_004102,hitting_time_expectation,symmetric_boundaries_zero_a,1,True,False,True,Let (S_n) be a simple symmetric random walk on...,{'expected_time': '9'},"{""expected_time"": 3}",{'expected_time': 3},False,non_binary
3,hitting_time_expectation_dev_004103,hitting_time_expectation,symmetric_boundaries_zero_a,1,False,False,False,Let (X_n) be a simple symmetric random walk on...,{'expected_time': '6'},"{""expected_time"": 3.0}",{'expected_time': 3.0},False,non_binary
4,hitting_time_expectation_dev_004104,hitting_time_expectation,symmetric_boundaries_zero_a,1,False,False,False,Let (S_n) be a simple symmetric random walk on...,{'expected_time': '15'},"{""expected_time"": 4}",{'expected_time': 4},False,non_binary


## Metrics

In [10]:
print(f"Overall accuracy: {df['correct'].mean():.3f} ({df['correct'].sum()}/{len(df)})")
display(df.groupby("family")["correct"].agg(["mean", "sum", "count"]).sort_index())
display(df.groupby("difficulty")["correct"].agg(["mean", "sum", "count"]).sort_index())
display(df.groupby("manual_variation")["correct"].agg(["mean", "sum", "count"]).sort_index())

Overall accuracy: 0.300 (18/60)


,mean,sum,count
family,,,
hitting_time_expectation,0.000000,0,15
martingale_verification,0.733333,11,15
optional_stopping_validity,0.333333,5,15
stopped_process_expectation,0.133333,2,15


,mean,sum,count
difficulty,,,
1,0.3,6,20
2,0.4,8,20
3,0.2,4,20


,mean,sum,count
manual_variation,,,
False,0.416667,10,24
True,0.222222,8,36


## Save Results

In [11]:
result_dir = PROJECT_ROOT / "results" / "baselines" / "qwen3_1_7b_test_closed_book"
metrics = summarize_accuracy(df)
metrics.update({
    "model": MODEL_NAME,
    "provider": "local_transformers",
    "dataset": "benchmark/data/test/*.jsonl",
    "grading_policy": GRADING_POLICY,
})

outputs_path, metrics_path, csv_path = save_results(rows, result_dir, metrics)
outputs_path, metrics_path, csv_path

(PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/baselines/qwen3_1_7b_test_closed_book/outputs.jsonl'),
 PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/baselines/qwen3_1_7b_test_closed_book/metrics.json'),
 PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/baselines/qwen3_1_7b_test_closed_book/outputs.csv'))

## Inspect Mistakes

In [12]:
df.loc[~df["correct"], ["id", "family", "problem_type", "difficulty", "canonical_answer", "predicted_answer", "raw_output"]].head(20)

,id,family,problem_type,difficulty,canonical_answer,predicted_answer,raw_output
0,hitting_time_expectation_dev_004100,hitting_time_expectation,symmetric_boundaries_zero_a,1,{'expected_time': '21'},{'expected_time': '2'},"{""expected_time"": ""2""}"
1,hitting_time_expectation_dev_004101,hitting_time_expectation,symmetric_boundaries_zero_a,1,{'expected_time': '6'},{'expected_time': '2.5'},"{""expected_time"": ""2.5""}"
2,hitting_time_expectation_dev_004102,hitting_time_expectation,symmetric_boundaries_zero_a,1,{'expected_time': '9'},{'expected_time': 3},"{""expected_time"": 3}"
3,hitting_time_expectation_dev_004103,hitting_time_expectation,symmetric_boundaries_zero_a,1,{'expected_time': '6'},{'expected_time': 3.0},"{""expected_time"": 3.0}"
4,hitting_time_expectation_dev_004104,hitting_time_expectation,symmetric_boundaries_zero_a,1,{'expected_time': '15'},{'expected_time': 4},"{""expected_time"": 4}"
5,hitting_time_expectation_dev_004200,hitting_time_expectation,symmetric_shifted_boundaries,2,{'expected_time': '3'},{'expected_time': '2.0'},"{""expected_time"": ""2.0""}"
6,hitting_time_expectation_dev_004201,hitting_time_expectation,symmetric_shifted_boundaries,2,{'expected_time': '9'},{'expected_time': 3.0},"{""expected_time"": 3.0}"
7,hitting_time_expectation_dev_004202,hitting_time_expectation,symmetric_shifted_boundaries,2,{'expected_time': '5'},{'expected_time': '6'},"{""expected_time"": ""6""}"
8,hitting_time_expectation_dev_004203,hitting_time_expectation,symmetric_shifted_boundaries,2,{'expected_time': '12'},{'expected_time': '4.0'},"<answer>{""expected_time"": ""4.0""}</answer>"
9,hitting_time_expectation_dev_004204,hitting_time_expectation,symmetric_shifted_boundaries,2,{'expected_time': '4'},{'expected_time': 3.0},"{""expected_time"": 3.0}"
